In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *
from biked_commons.benchmark_models.generative_models import utils


In [2]:
import importlib
importlib.reload(utils)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)
data_tens = torch.tensor(data.values, dtype=torch.float32, device=device)

scaler = utils.TorchScaler(data_tens)

data_tens = scaler.scale(data_tens)

continuous_conditioning = utils.sample_continuous(len(data_tens))
aux_fn = utils.get_diversity_loss_fn(scaler, 0.1, 0.1, 10, device=device)

In [3]:
# train_params = batch_size, disc_lr, gen_lr, noise_dim, num_epochs, n_hidden, layer_size
noise_dim = 10
batch_size = 64
train_params = (batch_size, 0.0003, 0.0003, noise_dim, 1, 2, 128)
D, G, generate_fn = utils.train_model(data_tens, "GAN", train_params, aux_fn, device)


c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\benchmark_models\generative_models\../../..\biked_commons\benchmark_models\generative_models\utils.py:434: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data = torch.tensor(data).float()
  0%|          | 0/70 [00:00<?, ?it/s]c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\benchmark_models\generative_models\../../..\biked_commons\benchmark_models\generative_models\utils.py:79: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShap

In [7]:
def predict(cond_test):
    cond_test = cond_test.to(device)

    predictions = []
    #use batch size
    for i in range(0, len(cond_test), batch_size):
        batch = cond_test[i:i + batch_size]
        with torch.no_grad():
            pred = generate_fn(D, G, batch, noise_dim, device=device)
            pred = scaler.unscale(pred)
            predictions.append(pred)
    predictions = torch.concat(predictions, axis=0)
    return predictions


In [10]:
def evaluate_model(predict_fn):
    continuous_condition = utils.sample_continuous(10000, split="test", randomize=False)
    condition = utils.parse_continuous_condition(continuous_condition)
    main_scorer = construct_scorer(MainScores, get_standard_evaluations(device), data.columns)
    detailed_scorer = construct_scorer(DetailedScores, get_standard_evaluations(device), data.columns)
    predictions = predict_fn(continuous_condition).detach()
    main_scores = main_scorer(predictions, condition)
    detailed_scores = detailed_scorer(predictions, condition)
    return main_scores, detailed_scores


In [11]:
main_scores, detailed_scores = evaluate_model(predict)

In [12]:
print("Main Scores:")
print(main_scores)
print("Detailed Scores:")
print(detailed_scores)

Main Scores:
Hypervolume                     0.000000
Constraint Satisfaction Rate    0.866667
Maximum Mean Discrepancy        0.062919
dtype: float64
Detailed Scores:
Min Objective Score: Drag Force                                                                              27.732685
Min Objective Score: Knee Angle Error                                                                       197.293720
Min Objective Score: Hip Angle Error                                                                        805.295530
Min Objective Score: Arm Angle Error                                                                        842.159670
Min Objective Score: Cosine Similarity to Embedding                                                           0.286874
Min Objective Score: Mass                                                                                    14.987815
Min Objective Score: Planar Compliance                                                                      115.509544